### 1. Import Libraries

In [1]:
import pandas as pd

### 2. Load and Consolidate Data

In [2]:
# Define the path to the Excel file
file_path = '../../data/source/leptospirosis.xlsx'

# Load the Excel file
leptospirosis_source_file = pd.ExcelFile(file_path)

# Initialize an empty list to store individual DataFrames from each sheet
all_data = []

# Loop through each sheet name
for sheet_name in leptospirosis_source_file.sheet_names:
    # Read the sheet into a pandas DataFrame, using the first column as district names
    df = leptospirosis_source_file.parse(sheet_name)
    
    # Extract the year from the sheet name
    year = sheet_name
    

    # dynamically select the district column and then using the remaining columns for melt.
    district_col = df.columns[0]
    df_melted = df.melt(id_vars=[district_col], var_name='Month', value_name='Cases')
    
    # Rename the district column to 'District'
    df_melted = df_melted.rename(columns={district_col: 'District'})
    
    # Add the 'Year' column
    df_melted['Year'] = year
    
    # Append the processed DataFrame to the list
    all_data.append(df_melted)


# Concatenate all DataFrames in the list into a single DataFrame
consolidated_leptospirosis_df = pd.concat(all_data, ignore_index=True)

print(consolidated_leptospirosis_df.head())

       District Month  Cases  Year
0        Ampara   Jan      0  2007
1  Anuradhapura   Jan      6  2007
2       Badulla   Jan      5  2007
3    Batticaloa   Jan      0  2007
4       Colombo   Jan     12  2007


### 3. Processing

 - Map official district names
 - Drop invalid districts

##### 3.1. District Mapping

In [3]:
consolidated_leptospirosis_df[
    (consolidated_leptospirosis_df["Year"] == "2011") 
]


,District,Month,Cases,Year
1236,Ampara (pottuvil),Jan,8,2011
1237,Anuradhapura,Jan,25,2011
1238,Badulla,Jan,2,2011
1239,Batticaloa,Jan,0,2011
1240,Colombo,Jan,31,2011
...,...,...,...,...
1531,Polonnaruwa,Dec,4,2011
1532,Puttalam,Dec,3,2011
1533,Ratnapura,Dec,42,2011
1534,Trincomalee,Dec,6,2011


In [4]:
district_mapping = {

    # --------------------
    # AMPARA
    # --------------------
    "Ampara": "Ampara",
    "Ampara (pottuvil)": "Ampara",

    # --------------------
    # ANURADHAPURA
    # --------------------
    "Anuradhapura": "Anuradhapura",
    "Anuradhapur": "Anuradhapura",
    "Apura": "Anuradhapura",

    # --------------------
    # BADULLA
    # --------------------
    "Badulla": "Badulla",

    # --------------------
    # BATTICALOA
    # --------------------
    "Batticaloa": "Batticaloa",

    # --------------------
    # COLOMBO
    # --------------------
    "Colombo": "Colombo",

    # --------------------
    # GALLE
    # --------------------
    "Galle": "Galle",

    # --------------------
    # GAMPAHA
    # --------------------
    "Gampaha": "Gampaha",
    "Gampaha (katunayake)": "Gampaha",

    # --------------------
    # HAMBANTOTA
    # --------------------
    "Hambantota": "Hambantota",

    # --------------------
    # JAFFNA
    # --------------------
    "Jaffna": "Jaffna",

    # --------------------
    # KALMUNAI (NOT A DISTRICT)
    # --------------------
    "Kalmunai": None,
    "Kalmunne": None,

    # --------------------
    # KALUTARA
    # --------------------
    "Kalutara": "Kalutara",

    # --------------------
    # KANDY
    # --------------------
    "Kandy": "Kandy",

    # --------------------
    # KEGALLE
    # --------------------
    "Kegalle": "Kegalle",

    # --------------------
    # KILINOCHCHI
    # --------------------
    "Kilinochchi": "Kilinochchi",

    # --------------------
    # KURUNEGALA
    # --------------------
    "Kurunegala": "Kurunegala",

    # --------------------
    # MANNAR
    # --------------------
    "Mannar": "Mannar",

    # --------------------
    # MATALE
    # --------------------
    "Matale": "Matale",

    # --------------------
    # MATARA
    # --------------------
    "Matara": "Matara",

    # --------------------
    # MONARAGALA
    # --------------------
    "Monaragala": "Monaragala",
    "Moneragala": "Monaragala",

    # --------------------
    # MULLAITIVU
    # --------------------
    "Mulativu": "Mullaitivu",
    "Mullaitivu": "Mullaitivu",

    # --------------------
    # NUWARA ELIYA
    # --------------------
    "N Eliya": "Nuwara Eliya",
    "Nuwara": "Nuwara Eliya",
    "Nuwara Eliya": "Nuwara Eliya",
    "NuwaraEliya": "Nuwara Eliya",


    # --------------------
    # POLONNARUWA
    # --------------------
    "Polonnaruwa": "Polonnaruwa",

    # --------------------
    # PUTTLAM
    # --------------------
    "Puttalam": "Puttalam",

    # --------------------
    # RATNAPURA
    # --------------------
    "Ratnapura": "Ratnapura",

    # --------------------
    # TRINCOMALEE
    # --------------------
    "Trincomalee": "Trincomalee",

    # --------------------
    # VAVUNIYA
    # --------------------
    "Vavuniya": "Vavuniya"
}


In [5]:
consolidated_leptospirosis_df["District"] = consolidated_leptospirosis_df["District"].map(district_mapping)

# Drop non-districts
consolidated_leptospirosis_df = consolidated_leptospirosis_df.dropna(subset=["District"]).reset_index(drop=True)


##### 3.2. Month Mapping

In [6]:
month_map = {
    "Jan": 1, "Feb": 2, "Mar": 3, "Apr": 4,
    "May": 5, "Jun": 6, "Jul": 7, "Aug": 8,
    "Sep": 9, "Oct": 10, "Nov": 11, "Dec": 12
}


In [7]:
consolidated_leptospirosis_df["Month"] = consolidated_leptospirosis_df["Month"].map(month_map)


#### 3.3. Data Quality Check

In [8]:
district_count_per_month = (
    consolidated_leptospirosis_df
    .groupby(["Year", "Month"])["District"]
    .nunique()
    .reset_index(name="District_Count")
)


month_count_per_district = (
    consolidated_leptospirosis_df
    .groupby("District")[["Month", "Year"]]
    .apply(lambda x: x.drop_duplicates().shape[0])
    .reset_index(name="Month_Count")
)




In [9]:
display(month_count_per_district)

,District,Month_Count
0,Ampara,216
1,Anuradhapura,216
2,Badulla,216
3,Batticaloa,216
4,Colombo,216
5,Galle,216
6,Gampaha,216
7,Hambantota,216
8,Jaffna,216
9,Kalutara,216


In [10]:
##Check whether all the data are available
district_count_per_month.to_csv('../../data/processed/data_quality_check/district_count_per_month.csv',
                index=False)

##Check whether all the data are available
month_count_per_district.to_csv('../../data/processed/data_quality_check/month_count_per_district.csv',
                index=False)

### Save Processed Data
 - data\processed

In [11]:
##saved as a new csv
consolidated_leptospirosis_df.to_csv('../../data/processed/1_leptospirosis_processed.csv',
                index=False)